Week 11 (07/05/2025) - Multilayer perceptrons

An artificial neural network is a learning structure that is inspired by the human brain. <br>
The simplest ANN model is the neuron, which provided a weighted linear sum of the features, although this structure is not very feasible in reality as it always requires manual computations for all thresholds. <br>
For this reason, most modern ANN models are based on the perceptron, which instead computes the threshold using a dedicated activation function, although this model only works with linear datasets.

A multilayer perceptron typically learns by repeating feedforwarding, which consists in estimating the correctness of the predictions through the mean square error (computed via loss function), and backpropagation, which consists in finding better weights via stochastic gradient descent, until convergence.

In [ ]:
# Pytorch implementation of a multilayer perceptron.
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            # Define the neural network's structure.
            nn.Linear(28*28,5), # connect the input layer (one input node per feature) to the hidden layer (five nodes)
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,5), # connect the first hidden layer to the second hidden layer
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,5), # connect the second hidden layer to the third hidden layer
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,10) # connect the third hidden layer to the output layer (ten nodes, one for each class)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits # note that the output is a probbaility tensor, so something of type logits = [0.1, 0.0, ..., 0.6]

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 2 # check the whole dataset twice
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass')

# Define the optimizer for gradient descent.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # perform stochastic gradient descent on the parameters